# IMQCAM DMS API Endpoints

This notebook calls each REST endpoint on the IMQCAM DMS directly using the Girder client, showing what parameters each accepts and what the raw response looks like.

| Endpoint | Description |
|---|---|
| `GET /form` | List the form schemas registered on the DMS |
| `GET /entry` | Fetch form entries — the actual experimental records |
| `GET /deposition` | List registered IGSNs and their DataCite metadata |
| `GET /sample` | List tracked samples |
| `GET /sample/id` | Fetch one sample with its full event history |

For a field-by-field catalog of every form, see `02_form_catalog.ipynb`.

## Setup

Authentication needs a `GIRDER_API_KEY`. Put it in a `.env` file at the repo root — `get_client()` reads it from there and pins the host to `https://data.imqcam.org/api/v1`.

The `REPO_ROOT` lookup walks up from the working directory until it finds `imqcam.py`, so the notebook works the same whether it is run from the repo root, from `tutorials/`, or from `tutorials/analysis/`.

In [1]:
import json
import sys
from pathlib import Path

REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "imqcam.py").exists())
sys.path.insert(0, str(REPO_ROOT))

from imqcam import get_client

client = get_client()
print(f"Connected to {client.urlBase}")

Connected to https://data.imqcam.org/api/v1/


## `GET /form`

A form is a schema. Every experimental record on the DMS belongs to exactly one, identified by its `_id` — which is what entries call `formId`.

There are nine.

In [2]:
forms = client.get("form", parameters={"limit": 1000})

print(f"{len(forms)} forms registered\n")
for form in sorted(forms, key=lambda f: f["name"]):
    print(f"  {form['_id']}  {form['name']}")

9 forms registered

  6970da157f6ebb8fb320705c  AM Build Parameters
  68922e35f5b193b7d3e07f5b  Archival ULI Build (simplified)
  68922e94f5b193b7d3e07f5c  Four-point flexural test
  69fa1d0872a32de5fe95028b  Fractography Record
  68ed0eb251d2cf4a1aa1d14f  Micromechanical Simulation Data
  67d39472366ec49ab59dd4db  NASA Ti64 30um Layers Data
  66425a71b18fa1c426e93aa0  Printer Build
  663e6d21b18fa1c426e939ab  Raw Powder Details
  69fa18be514cf501621588cc  Sample Heat Treatment Record


## `GET /entry`

The main data endpoint. Called bare it returns entries from every form at once, but it takes a full set of query parameters — filtering, searching and sorting all happen server-side.

| Parameter | Default | Description |
|---|---|---|
| `formId` | — | Restrict to a single form. Do this rather than fetching everything and filtering in pandas |
| `query` | — | Regex matched against `field` |
| `field` | `sampleId` | Which field `query` searches |
| `limit` | 50 | Page size |
| `offset` | 0 | Pagination offset |
| `sort` | `created` | Field to sort by |
| `sortdir` | 1 | 1 ascending, -1 descending |

In [3]:
# One page, to look at the shape of a single record.
page = client.get("entry", parameters={"limit": 3})
print(f"Returned {len(page)} entries")
print(json.dumps(page[0], indent=2, default=str)[:900])

Returned 3 entries
{
  "_id": "663e8182b18fa1c426e93a7b",
  "created": "2024-05-10T20:20:18.775000+00:00",
  "data": {
    "alloy": "Ti-6Al-4V",
    "batchInformation": [
      {
        "key": "original_power_id",
        "value": "ATI_Ti64_batch_1609_heat_9E64STD004"
      }
    ],
    "characteristics": {
      "maxSize": 53,
      "minSize": 15,
      "tested": true,
      "virginPercent": 0
    },
    "composition": [
      {
        "amount": 0,
        "element": "Ti",
        "isBalance": true,
        "method": "NA (for balance)",
        "unit": "%"
      },
      {
        "amount": 6.09,
        "element": "Al",
        "isBalance": false,
        "method": "WET (Inductively coupled plasma emission)",
        "unit": "%"
      },
      {
        "amount": 3.89,
        "element": "V",
        "isBalance": false,
        "method": "WET (Inductively coupled plasma emission)",
        "unit": "%"



### The entry envelope

Every entry, whatever form it came from, carries the same outer fields. Only `data` differs between forms.

- **`_id`** — the entry's own Girder id
- **`uniqueId`** — the human-readable key the form defines (a build id, a heat-treatment id, a sample IGSN)
- **`formId`** — which schema this entry belongs to
- **`created`, `updated`** — timestamps
- **`folderId`, `folders`, `files`** — attached Girder storage
- **`data`** — the form payload, and the only part whose shape varies

So a generic reader can always page `/entry`, group by `formId`, and treat `data` as the form-specific part.

In [4]:
envelope = {k: v for k, v in page[0].items() if k != "data"}
print("Envelope fields:")
for key, value in sorted(envelope.items()):
    print(f"  {key:12s} {type(value).__name__:6s} {str(value)[:60]}")

print(f"\nPayload (`data`) keys: {sorted(page[0].get('data', {}))}")

Envelope fields:
  _id          str    663e8182b18fa1c426e93a7b
  created      str    2024-05-10T20:20:18.775000+00:00
  files        list   ['663e8179b18fa1c426e93a79']
  folderId     str    663e6e1ab18fa1c426e939ae
  folders      list   []
  formId       str    663e6d21b18fa1c426e939ab
  uniqueId     str    PWD_Ti-6Al-4V_ATI_CMU_001_batch-1609_heat-9E64STD004
  updated      str    2024-05-10T20:20:18.775000+00:00

Payload (`data`) keys: ['alloy', 'batchInformation', 'characteristics', 'composition', 'extraInfo', 'files', 'purchaser', 'rawPowderID', 'sampleNumber', 'supplyCompany']


### Filtering by form

`formId` is applied server-side, so only the form you asked for crosses the wire. Fetching all 810 entries to keep 193 of them is wasted work that gets worse as the collection grows.

In [5]:
FLEXURAL = "68922e94f5b193b7d3e07f5c"

filtered = client.get("entry", parameters={"formId": FLEXURAL, "limit": 1000})
print(f"Server-side formId filter: {len(filtered)} entries")
print(f"All from the requested form: {all(e['formId'] == FLEXURAL for e in filtered)}")

Server-side formId filter: 193 entries
All from the requested form: True


### Three things to watch

- **`limit` is a ceiling, not pagination.** Asking for `limit=1000` returns *at most* 1000 records and gives no indication that it truncated. The collection is under that today, so the bug is latent — but it will start silently dropping data the moment it isn't. Page with `offset` until a short page comes back; `fetch_entries()` in `imqcam.py` does exactly this, and sorts explicitly while doing so, since offset paging over an unordered result set can repeat or skip rows between requests.
- **`data` payloads are not flat.** Several forms nest objects (`buildParameters.infillParameters.laserPower`) and several hold *lists* of repeated measurements (`tests`, `composition`). One entry does not mean one row — see `explode_records()`.
- **Some entries reference forms that `/form` does not return.** Their `formId` resolves to nothing, so a naive join drops them without complaint. `02_form_catalog.ipynb` reports these explicitly.

Paging properly gives the true total:

In [6]:
from imqcam import fetch_entries

entries = fetch_entries(client, page_size=100)
print(f"Total entries across all forms: {len(entries)}")

# Compare against a single large request -- equal today, divergent once the
# collection outgrows the ceiling.
single_request = client.get("entry", parameters={"limit": 1000})
print(f"Single `limit=1000` request:     {len(single_request)}")

# Per-form counts without ever fetching the whole collection.
print()
for form in sorted(forms, key=lambda f: f["name"])[:4]:
    n = len(fetch_entries(client, form_id=form["_id"], page_size=100))
    print(f"  {form['name']:34s} {n:3d}")

Total entries across all forms: 810


Single `limit=1000` request:     810



  AM Build Parameters                 27


  Archival ULI Build (simplified)    192


  Four-point flexural test           193


  Fractography Record                 72


## `GET /deposition`

Depositions are the IGSN registry: each one mints a sample identifier and carries DataCite metadata (DOI, creators, dates, titles). This is the spine that links records across forms.

In [7]:
depositions = client.get("deposition", parameters={"limit": 1000})
print(f"IGSNs registered: {len(depositions)}\n")

example = depositions[0]
print(f"IGSN: {example['igsn']}")
print(f"DOI:  {example.get('metadata', {}).get('doi')}")
print(f"State: {example.get('state')}")
print(f"\nMetadata keys: {sorted(example.get('metadata', {}))}")

IGSNs registered: 696

IGSN: APLMAL00001
DOI:  10.82581/APLMAL00001
State: draft

Metadata keys: ['alternateIdentifiers', 'contributors', 'creators', 'dates', 'descriptions', 'doi', 'formats', 'fundingReferences', 'geoLocations', 'publicationYear', 'publisher', 'relatedIdentifiers', 'relatedItems', 'rightsList', 'schemaVersion', 'sizes', 'subjects', 'titles', 'types', 'url']


## `GET /sample` and `GET /sample/id`

Samples are the physical specimens. The list endpoint returns identity only; the event history — where a sample went and what was done to it — requires fetching each sample individually by id.

There is no bulk event endpoint, so building a full timeline is unavoidably one request per sample. `03_sample_tracking.ipynb` does this.

In [8]:
samples = client.get("sample", parameters={"limit": 1000})
print(f"Samples tracked: {len(samples)}")

# Most samples have no recorded events; walk until one does, so the example
# below actually shows an event payload.
SCAN_LIMIT = 60
detail, events = None, []
for sample in samples[:SCAN_LIMIT]:
    detail = client.get("sample/id", parameters={"id": sample["_id"]})
    events = detail.get("events", [])
    if events:
        break

print(f"First sample with events in the leading {SCAN_LIMIT}: {detail.get('name')} ({len(events)} events)")
if events:
    print(f"Event fields: {sorted(events[0])}")
    print(f"\nFirst event: {json.dumps(events[0], indent=2, default=str)[:400]}")

Samples tracked: 377


First sample with events in the leading 60: CMXMAL00010-032 (3 events)
Event fields: ['comment', 'created', 'creator', 'creatorName', 'eventType', 'location']

First event: {
  "comment": "Sample received at CWRU for 4PB testing.",
  "created": "2025-06-23T18:39:45.960000+00:00",
  "creator": "672e3ac5bc5bed1ca3ec022b",
  "creatorName": "Brett Ley",
  "eventType": "Sample Received",
  "location": "CWRU"
}
